# #23 · Record every estimate in a comparable schema

**Owner:** @gaurvsingh095  
**Issue:** [#23](https://github.com/Break-Through-Tech/Estee-Lauder-1B-measuring-customer-delight/issues/23)  
**Milestone:** #3 — Simple DiD: control selection & manual 2×2

## Goal

Make sure each treatment-effect estimate is saved in a shared, comparable schema so later milestones can compare:

- naive before/after
- manual 2×2 DiD
- later regression DiD
- later fixed-effects DiD
- later models with launch-context controls

The shared file is:

```text
results/estimates.csv
```

This notebook does not create a new estimate. It validates and displays the estimate registry created by issues #21 and #22.

In [1]:
import sys
sys.path.append("../../src")

import pandas as pd

import simple_did as sd

df = sd.load_panel()
launch = sd.launch_week(df)
print(f"Loaded {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Launch week: {launch.date()}")
df.head()

Loaded 728 rows × 16 columns
Launch week: 2026-04-06


,market,week_index,week_start,sessions,fragrance_orders,fragrance_revenue,conversion_rate,average_order_value,revenue_per_session,paid_media_index,promo_intensity,new_visitor_share,pilot_market,post_launch,digital_feature_available,weeks_from_launch
0,Australia,0,2025-01-06,69314,1870,147050.31,0.026979,78.64,2.1215,96.81,0.2081,0.3308,0,0,0,-65
1,Australia,1,2025-01-13,71010,2011,159108.26,0.028320,79.12,2.2406,100.78,0.2142,0.3870,0,0,0,-64
2,Australia,2,2025-01-20,71268,1954,150249.86,0.027418,76.89,2.1082,89.21,0.2005,0.3705,0,0,0,-63
3,Australia,3,2025-01-27,67993,1632,128219.61,0.024002,78.57,1.8858,104.77,0.2123,0.3519,0,0,0,-62
4,Australia,4,2025-02-03,68280,1952,161268.21,0.028588,82.62,2.3619,101.49,0.1982,0.3558,0,0,0,-61


## Step 1 — Confirm the expected schema

The schema is defined in `src/simple_did.py` as `sd.ESTIMATE_COLUMNS`. Every estimate row should use these columns.

In [2]:
expected_columns = sd.ESTIMATE_COLUMNS
expected_columns

['method',
 'treated',
 'control',
 'metric',
 'window',
 'estimate',
 'ci_low',
 'ci_high',
 'issue',
 'author',
 'recorded_at',
 'notes']

## Step 2 — Load the current estimate registry

In [3]:
estimates = sd.load_estimates()
estimates

,method,treated,control,metric,window,estimate,ci_low,ci_high,issue,author,recorded_at,notes
0,naive_before_after,United States,,revenue_per_session,all_pre_vs_post,0.342090,NaN,NaN,21,@gaurvsingh095,2026-09-22T18:50:58Z,Naive U.S. before/after change. Descriptive ba...
1,manual_2x2_did,United States,Canada,revenue_per_session,all_pre_vs_post,0.245914,NaN,NaN,22,@gaurvsingh095,2026-09-22T18:51:02Z,Manual 2x2 DiD ATT using Canada as control. Mo...


## Step 3 — Validate schema and identify current records

In [4]:
missing = [c for c in expected_columns if c not in estimates.columns]
extra = [c for c in estimates.columns if c not in expected_columns]

validation = pd.DataFrame([{
    "n_estimates_recorded": len(estimates),
    "schema_matches_expected": estimates.columns.tolist() == expected_columns,
    "missing_columns": missing,
    "extra_columns": extra,
}])

validation

,n_estimates_recorded,schema_matches_expected,missing_columns,extra_columns
0,2,True,[],[]


## Step 4 — Slide-ready summary of recorded estimates

In [5]:
summary_cols = ["issue", "method", "treated", "control", "metric", "window", "estimate", "notes"]
slide_ready = estimates[summary_cols].copy()
slide_ready["estimate"] = pd.to_numeric(slide_ready["estimate"], errors="coerce")
slide_ready.round({"estimate": 4})

,issue,method,treated,control,metric,window,estimate,notes
0,21,naive_before_after,United States,,revenue_per_session,all_pre_vs_post,0.3421,Naive U.S. before/after change. Descriptive ba...
1,22,manual_2x2_did,United States,Canada,revenue_per_session,all_pre_vs_post,0.2459,Manual 2x2 DiD ATT using Canada as control. Mo...


## Final answer for issue #23

The estimate registry provides one place to compare all current and future treatment-effect estimates. For this milestone, it should contain the naive U.S. before/after estimate from issue #21 and the manual 2×2 DiD estimate from issue #22.